In [1]:
from qiskit_ibm_runtime.fake_provider import FakeFez, FakeTorino, FakeMarrakesh, FakePerth
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit import QuantumCircuit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_aer import AerSimulator

In [ ]:
backend = FakeTorino()

coupling_map = backend.coupling_map
two_qubit_gate = 'ecr' if 'ecr' in backend.operation_names else 'cz'

error_map = {}
for edge in coupling_map.get_edges():

        error_map[edge] = backend.target[two_qubit_gate][edge].error


print("Error map:", error_map)

Error map: {(0, 1): 0.0017402852518782486, (1, 0): 0.0017402852518782486, (1, 2): 0.001548074406909461, (2, 1): 0.001548074406909461, (2, 3): 0.003201528503065487, (3, 2): 0.003201528503065487, (3, 4): 0.0040483808884482775, (3, 16): 0.0035023762389818636, (4, 3): 0.0040483808884482775, (4, 5): 0.0023648392641392735, (5, 4): 0.0023648392641392735, (5, 6): 0.001692163754732634, (6, 5): 0.001692163754732634, (6, 7): 0.0022869653902582443, (7, 6): 0.0022869653902582443, (7, 8): 0.0022223047094009907, (7, 17): 0.0026334033865881845, (8, 7): 0.0022223047094009907, (8, 9): 0.003635313584797739, (9, 8): 0.003635313584797739, (9, 10): 0.002285248593876854, (10, 9): 0.002285248593876854, (10, 11): 0.00244098492883299, (11, 10): 0.00244098492883299, (11, 12): 0.0012310019540017758, (11, 18): 0.0024962124357810755, (12, 11): 0.0012310019540017758, (12, 13): 0.0012392654545236303, (13, 12): 0.0012392654545236303, (13, 14): 0.0010363120130211234, (14, 13): 0.0010363120130211234, (14, 15): 0.0012396

In [10]:
def bfs_sum_error_depth(start, coupling_map, depth, qubit_scores):
    visited = set()
    queue = [(start, 0)]  # (node, current_depth)
    area_score = 1

    while queue:
        node, current_depth = queue.pop(0)
        if node not in visited and current_depth <= depth:
            visited.add(node)
            area_score *= qubit_scores[node]
            
            neighbor_indices = [n for n in coupling_map.neighbors(node)]

            for neighbor in neighbor_indices:
                queue.append((neighbor, current_depth + 1))
    
    
    
    return area_score

In [11]:
qubit_scores = {i: 1.0 for i in range(backend.num_qubits)}
# print("Initial qubit scores:", qubit_scores)

for edge, error in error_map.items():
    qubit_scores[edge[0]] *= (1 - error)
    qubit_scores[edge[1]] *= (1 - error)

best_area_queue = []

for i in range(backend.num_qubits):
    qubit_props = backend.properties().qubit_property(i)
    readout_error = qubit_props.get("readout_error", (None,))[0]
    qubit_scores[i] *= (1 - readout_error)
    qubit_scores[i] *= bfs_sum_error_depth(i, coupling_map, depth=2, qubit_scores=qubit_scores)

highest_score_qubit = max(qubit_scores, key=qubit_scores.get)
print("Highest score qubit:", highest_score_qubit, "with score:", qubit_scores[highest_score_qubit])
# print("Updated qubit scores:", qubit_scores)

Highest score qubit: 140 with score: 0.9598246793406818


In [12]:
print(qubit_scores)

highest_score_qubits = sorted(qubit_scores, key=qubit_scores.get, reverse=True)[:2]
print("Highest score qubits:", highest_score_qubits, "with scores:", [qubit_scores[q] for q in highest_score_qubits])

{0: 0.9553770062590394, 1: 0.9042583854411155, 2: 0.8013004719510023, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0, 8: 0.0, 9: 0.0, 10: 0.0, 11: 0.0, 12: 0.0, 13: 0.0, 14: 0.0, 15: 0.0, 16: 0.0, 17: 0.0, 18: 0.0, 19: 0.0, 20: 0.9171353682393316, 21: 0.0, 22: 0.0, 23: 0.0, 24: 0.0, 25: 0.0, 26: 0.0, 27: 0.0, 28: 0.0, 29: 0.0, 30: 0.0, 31: 0.0, 32: 0.0, 33: 0.0, 34: 0.0, 35: 0.0, 36: 0.0, 37: 0.0, 38: 0.0, 39: 0.0, 40: 0.0, 41: 0.0, 42: 0.0, 43: 0.0, 44: 0.0, 45: 0.0, 46: 0.0, 47: 0.0, 48: 0.0, 49: 0.0, 50: 0.0, 51: 0.0, 52: 0.0, 53: 0.0, 54: 0.0, 55: 0.0, 56: 0.0, 57: 0.0, 58: 0.0, 59: 0.0, 60: 0.6962068252145983, 61: 0.3234145636848916, 62: 0.0, 63: 0.0, 64: 0.0, 65: 0.0, 66: 0.0, 67: 0.0, 68: 0.0, 69: 0.0, 70: 0.0, 71: 0.0, 72: 0.0, 73: 0.0, 74: 0.0, 75: 0.0, 76: 0.0, 77: 0.0, 78: 0.0, 79: 0.0, 80: 0.0, 81: 0.0, 82: 0.0, 83: 0.0, 84: 0.0, 85: 0.0, 86: 0.0, 87: 0.0, 88: 0.0, 89: 0.0, 90: 0.0, 91: 0.0, 92: 0.0, 93: 0.0, 94: 0.0, 95: 0.0, 96: 0.0, 97: 0.0, 98: 0.0, 99: 0.0, 100: 0.0, 101: 0.0,